In [26]:
import torch

print(torch.cuda.is_available())

True


In [27]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn as nn

In [28]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [29]:
train_dataset = datasets.ImageFolder(
    root=r"C:\Desktop\Dog_Cat_Cls\dataset\train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=r"C:\Desktop\Dog_Cat_Cls\dataset\test",
    transform=transform

)

val_dataset = datasets.ImageFolder(
    root=r"C:\Desktop\Dog_Cat_Cls\dataset\val",
    transform=transform
)

In [30]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [31]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [32]:
class CNN(nn.Module):
    def __init__(self, input_features=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),

        )

        self.fc = nn.Sequential(
            nn.Flatten(),

            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, num_classes)


        )


    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)

        return x

In [33]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [34]:
model = CNN(input_features=3, num_classes=2).to(device)

In [35]:
import torch.optim as optim
lr = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [36]:
for epoch in range(epochs):
    model.train()

    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        y_pred = model(images)

        optimizer.zero_grad()

        loss = criterion(y_pred, labels)

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()
    print(total_loss/len(train_loader))    


0.7643180415034294
0.37837405943057756
0.22712851654399524
0.15689201923933896
0.11584163596853614
0.16611716144887562
0.06823639408685267
0.04983598120849241
0.02297137505427765
0.014099780304108704


In [37]:
model.eval()

CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (9): ReLU()
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=Fa

In [38]:
total = 0
correct = 0

with torch.no_grad():
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)     

0.9971428571428571


In [39]:
total = 0
correct = 0

with torch.no_grad():
    for features, labels in val_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)     

0.99


In [40]:
total = 0
correct = 0

with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)     

0.925


In [42]:
torch.save(
    model.state_dict(),
    "dog_cat_cnn.pth"
)

In [46]:
from PIL import Image
import torch
import torch.nn.functional as F

# Load model
model = CNN(input_features=3, num_classes=2)

model.load_state_dict(
    torch.load("dog_cat_cnn.pth")
)

model = model.to(device)

model.eval()

# Class names
classes = ['cat', 'dog']

# Load image
image = Image.open(r"C:\Desktop\Dog_Cat_Cls\dataset\test\cat\00350-200124660.png")

# Apply transforms
image = transform(image)

# Add batch dimension
image = image.unsqueeze(0)

# Move to GPU/CPU
image = image.to(device)

# Prediction
with torch.no_grad():

    outputs = model(image)

    probabilities = F.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

print("Prediction:", classes[predicted.item()])

print("Confidence:", confidence.item())

Prediction: cat
Confidence: 1.0


In [47]:

from PIL import Image
import torch
import torch.nn.functional as F

# Load model
model = CNN(input_features=3, num_classes=2)

model.load_state_dict(
    torch.load("dog_cat_cnn.pth")
)

model = model.to(device)

model.eval()

# Class names
classes = ['cat', 'dog']

# Load image
image = Image.open(r"C:\Desktop\Dog_Cat_Cls\dataset\test\dog\00850-3846169012.png")

# Apply transforms
image = transform(image)

# Add batch dimension
image = image.unsqueeze(0)

# Move to GPU/CPU
image = image.to(device)

# Prediction
with torch.no_grad():

    outputs = model(image)

    probabilities = F.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

print("Prediction:", classes[predicted.item()])

print("Confidence:", confidence.item())

Prediction: dog
Confidence: 1.0
